In [1]:
from hat.registry import build_from_registry
from hat.utils.config import Config
#from plugin import *

#config = Config.fromfile("/home/users/xingyu.zhang/workspace/dataset_hat/projects/end2end/configs/dataloader.py")

/home/users/xingyu.zhang/miniconda3/envs/e2e_hat/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-18 16:18:58,226	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# get clip list:
from hat.data.datasets.multi_view_msg_dataset import (
    LmdbAnnoReader,
    LmdbSyncIndexReader,
    LmdbTagReader,
    LmdbRawDataReader
)
from hat.utils.bucket import get_bucket_client
bucket_cli = get_bucket_client()

sync_path = 'dmpv2://monotd_e2e_dev/sync_msg_lmdb/yun01.du/e2e_lmdb_data_datasetv6_0616/data/e2e_lmdb_data_datasetv6_0616_train_part_0/clip_dynamic_static/v1.2.2/e2e_dynamic_static/20240616_025645_312440/sync'
sync_path = bucket_cli.url_to_local(sync_path)
sync_reader = LmdbSyncIndexReader(lmdb_url=sync_path)

In [11]:
from tqdm import tqdm
from hat.utils.bucket import get_bucket_client


data_path = "dmpv2://monotd_e2e_dev/sync_msg_lmdb/yun01.du/e2e_lmdb_data_datasetv6_0616/data/e2e_lmdb_data_datasetv6_0616_train_part_0/clip_dynamic_static/v1.2.2/e2e_dynamic_static/20240616_025645_312440/data"
data_path = bucket_cli.url_to_local(data_path)
data_reader = LmdbRawDataReader(lmdb_url=data_path,camera_topics=['5', '2', '0', '3', '1', '4', '10'])



anno_path = 'dmpv2://monotd_e2e_dev/sync_msg_lmdb/yun01.du/datasetv6_0616with_obs_traj_0626/data/e2e_lmdb_data_datasetv6_0616_train_part_0_anno_incremental_part_0/clip_dynamic_static/v1.2.2/e2e_incremental_packing/20240626_221504_654504/anno'
anno_path = bucket_cli.url_to_local(anno_path)
anno_reader = LmdbAnnoReader(lmdb_url=anno_path,camera_topics=['5', '2', '0', '3', '1', '4', '10'])

json_data = []

for i in tqdm(range(len(sync_reader))):

    if i == 1:
        break

    one_seq_timestamp = []
    one_seq_filename = []
    one_seq_ego_pose = []

    json_one_seq = {}


    clip_msg = sync_reader.read(i)
    
    len_clip = len(clip_msg.messages)


    anno_res = anno_reader.read(clip_msg)
    anno_messages = anno_res.get_messages()


    #data_message = data_reader.read_dict(clip_msg)


    for k in range(len_clip):
        objects = anno_messages[k].get_messages()
        ego_pose = (objects[1].__dict__['ego_poses'][0].location[0],objects[1].__dict__['ego_poses'][0].location[1])
        one_seq_ego_pose.append(ego_pose)

        
        #one_seq_filename.append(bucket_cli.url_to_local(data_messages[k].get_messages()[2].image.url))
        #one_seq_timestamp.append(data_messages[k].get_messages()[2].image.name)

    json_one_seq['seq_ego_poses'] = one_seq_ego_pose
    #json_one_seq['seq_timestamp'] = one_seq_timestamp
    #json_one_seq['seq_filename'] = one_seq_filename
    json_one_seq['len_of_clip'] = len_clip

    json_data.append(json_one_seq)


  0%|          | 0/1000 [00:13<?, ?it/s]


AttributeError: 'LmdbRawDataReader' object has no attribute 'read_dict'

In [6]:
json_data[0]

{'seq_ego_poses': [(-10709.552734375, 24615.22265625),
  (-10708.0478515625, 24616.00390625),
  (-10706.5439453125, 24616.78515625),
  (-10705.0419921875, 24617.56640625),
  (-10703.541015625, 24618.34765625),
  (-10702.0498046875, 24619.12890625),
  (-10700.55078125, 24619.91015625),
  (-10699.0615234375, 24620.69140625),
  (-10697.55859375, 24621.47265625),
  (-10696.05859375, 24622.25390625),
  (-10694.564453125, 24623.033203125),
  (-10693.0703125, 24623.814453125),
  (-10691.56640625, 24624.59765625),
  (-10690.0615234375, 24625.37890625),
  (-10688.5556640625, 24626.16015625),
  (-10687.052734375, 24626.939453125),
  (-10685.548828125, 24627.71875),
  (-10684.0439453125, 24628.498046875),
  (-10682.5400390625, 24629.275390625),
  (-10681.0361328125, 24630.056640625),
  (-10679.5341796875, 24630.830078125),
  (-10678.0166015625, 24631.609375),
  (-10676.5068359375, 24632.390625),
  (-10674.9970703125, 24633.16796875),
  (-10673.4990234375, 24633.931640625),
  (-10671.99609375, 246

In [ ]:
anno_path = 'dmpv2://monotd_e2e_dev/sync_msg_lmdb/yun01.du/datasetv6_0616with_obs_traj_0626/data/e2e_lmdb_data_datasetv6_0616_train_part_0_anno_incremental_part_0/clip_dynamic_static/v1.2.2/e2e_incremental_packing/20240626_221504_654504/anno'
anno_path = bucket_cli.url_to_local(anno_path)
anno_reader = LmdbAnnoReader(lmdb_url=anno_path,camera_topics=['5', '2', '0', '3', '1', '4', '10'])
anno_res = anno_reader.fill_frames(clip_msg=clip_msg)

In [ ]:
ego_poses = []
anno_messages = anno_res.get_messages()
print(len(anno_messages))
objects = anno_messages[2].get_messages()
len(objects)
location = (objects[1].__dict__['ego_poses'][0].location[0],objects[1].__dict__['ego_poses'][0].location[1])
location
#objects[1]

In [97]:
data_path = "dmpv2://monotd_e2e_dev/sync_msg_lmdb/yun01.du/e2e_lmdb_data_datasetv6_0616/data/e2e_lmdb_data_datasetv6_0616_train_part_0/clip_dynamic_static/v1.2.2/e2e_dynamic_static/20240616_025645_312440/data"
data_path = bucket_cli.url_to_local(data_path)
data_reader = LmdbRawDataReader(lmdb_url=data_path,camera_topics=['0','1','2','3','4','5', '10'])
image_res = data_reader.fill_frames(anno_res)
#imgs = img.get_messages()
#print(imgs[0])

In [ ]:
len(image_res.get_messages())

In [92]:

objects = imgs[0].get_messages()
# import cv2
# type(objects[2].image.url)
# # 读取 JPG 图像
# image = cv2.imread(objects[2].image.url)

from PIL import Image
path = bucket_cli.url_to_local(objects[2].image.url)
# 打开 JPG 图像
image = Image.open(path)

# 显示图像
image.save('1.jpg')
# # 显示图像
# cv2.imshow("Image", image)

In [ ]:
image_res[0]

In [88]:
# format json

json_data = []
for 
one_seq_data = []
ego_poses = []
lens_


In [ ]:
#get json & load & show imge;
#测试读取图像时间：
